# Tide modelling using [`eo-tides`](https://geoscienceaustralia.github.io/eo-tides/) in [Piksel Sandbox](https://sandbox.staging.pik-sel.id/)

Utilising [`eo-tides`](https://geoscienceaustralia.github.io/eo-tides/) modules in [Piksel Sandbox](https://sandbox.staging.pik-sel.id/) to generate tide heights and combining tides with satellite data using tide models. The use cases and informations in this notebook are from [eo-tides](https://geoscienceaustralia.github.io/eo-tides/) and [piksel-notebooks](https://github.com/piksel-ina/piksel-notebooks) 

## Getting started

### Import

In [ ]:
import os
import pandas as pd
from ipyleaflet import basemaps
from urllib.parse import urlparse

import dask.config
from dask.distributed import Client, LocalCluster
from datacube import Datacube
from eo_tides.eo import tag_tides
from eo_tides.eo import pixel_tides
from eo_tides.model import model_tides
from eo_tides.utils import list_models
from odc.geo.geom import BoundingBox

from odc.stac import configure_s3_access

### Setting up inputs

In [ ]:
TIDE_MODEL_DIRECTORY = '/home/jovyan/data/coastlines/tide_models'

Storing initial variables inside a dictionary.

In [ ]:
vars_in = {
    'date_range': ('2025-07-01', '2025-07-31'),
    'coord1': (-4.027971562907108, 116.04213693831477),
    'loc_name1': 'IBT_jetty',
    'coord2': (-4.081853014790685, 116.09078930089663),
    'loc_name2': 'tj_selayar',
    'tide_model': 'INATIDES',
    'product': ['s2_l2a', 'ls5_c2l2_sr', 'ls7_c2l2_sr', 'ls8_c2l2_sr', 'ls9_c2l2_sr'],
    'measurements': ['red', 'green', 'blue', 'nir08'],
    'output_crs': 'utm', # try EPSG:4326 or EPSG:6933
    'dask_chunk': {'time': 1, 'x': 520, 'y': 520},
}

## Modelling tides

Demonstrate using `model_tides` function from `eo_tides.model` module to model tide heights.

Verify if the tide model is available in the directory.

In [ ]:
list_models(directory=TIDE_MODEL_DIRECTORY, show_supported=True)

Setting up time using pandas and store it in `vars_in`

In [ ]:
vars_in.update({
    'time': pd.date_range(
        start=vars_in['date_range'][0],
        end=vars_in['date_range'][1],
        freq='1h', # hourly frequency
    )
})

Generate hourly tide heights at 2 locations

In [ ]:
tide_df = model_tides(
    x=[vars_in['coords1'][1], vars_in['coords2'][1]],
    y=[vars_in['coords1'][0], vars_in['coords2'][0]],
    time=vars_in['time'],
    directory=TIDE_MODEL_DIRECTORY,
    model=vars_in['tide_model'],
)

tide_df

## Combining tides with satellite data

Combining tides with satellite data using `tag_tides` and `pixel-tides` functions from `eo_tides.eo`.

Comparison of `tag_tides` and `pixel_tides` 

| [`tag_tides`](../../api/#eo_tides.eo.tag_tides)| [`pixel_tides`](../../api/#eo_tides.eo.pixel_tides)|
|------------------------------------------------|----------------------------------------------------|
| Assigns a single tide height to each timestep/satellite image| Assigns a tide height to every individual pixel through time to capture spatial tide dynamics|
| 🔎 Ideal for local or site-scale analysis| 🌏 Ideal for regional to global-scale coastal product generation|
| ✅ Fast, low memory use| ❌ Slower, higher memory use|
| ❌ Single tide height per image can produce artefacts in complex tidal regions | ✅ Produce spatially seamless results across large extents by applying analyses at the pixel level |

In [ ]:
# Configure AWS
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'

if 'AWS_NO_SIGN_REQUEST' in os.environ:
    del os.environ['AWS_NO_SIGN_REQUEST']

configure_s3_access(requester_pays=True)

# Connect to the Datacube
dc = Datacube(app='coastlines')

In [ ]:
# Set up Dask
cluster = LocalCluster(n_workers=8, threads_per_worker=16, memory_limit='30GB')

dashboard_url = cluster.dashboard_link
port = urlparse(dashboard_url).port

jupyterhub_user = os.environ.get('JUPYTERHUB_USER')
dask.config.set(
    **{'distributed.dashboard.link': f'/user/{jupyterhub_user}/proxy/{port}/status'}
)

client = Client(cluster)
client

In [ ]:
# Insert coordinates of the opposing corners on the area of interest
lat1, lon1 = -4.111677506225011, 116.10987348008965
lat2, lon2 = -3.9017865592515095, 116.00756326944374

# switch lat and long so that 1 is always less than 2
if lat1 > lat2:
    lat1, lat2 = lat2, lat1
if lon1 > lon2:
    lon1, lon2 = lon2, lon1

vars_in.update({
    'date_range': ('2024', '2025'),
    'area': BoundingBox(lon1, lat1, lon2, lat2),
    'product': 's2_l2a', # Sentinel-2
    'measurements': ['red', 'green', 'blue', 'nir08'],
    'resolution': 10,
    'max_cloud_cover': 15, # percentage
    'output_crs': 'utm', # try EPSG:4326 or EPSG:6933
    'resampling': 'cubic',
    'dask_chunk': {'time': 1, 'x': 520, 'y': 520},
})

vars_in['area'].explore(tiles=basemaps.Esri.WorldImagery)

In [ ]:
ds = dc.find_datasets(
    product=vars_in['product'],
    time=vars_in['date_range'],
    longitude=,
    latitude=,
    cloud_cover=(0, vars_in['max_cloud_cover']),
)

print(f'Found {len(ds)} Sentinel-2 datasets')

data = dc.load(
    datasets=ds,
    longitude=(vars_in['area'].left, vars_in['area'].right),
    latitude=(vars_in['area'].bottom, vars_in['area'].top),
    resolution=vars_in['resolution'],
    measurements=vars_in['measurements'],
    output_crs=vars_in['output_crs'],
    group_by='solar_day',
    dask_chunks=vars_in['dask_chunk'],
    resampling=vars_in['resampling'],
    driver='rio',
)